# 🧪 Test Runner - Comprehensive Test Suite

This notebook provides an interactive way to run the SkillUP test suite with visual feedback.

## Quick Navigation
* [Run All Unit Tests](#run-all-unit-tests)
* [Run Tests by Module](#run-tests-by-module)
* [Smoke Tests](#smoke-tests)
* [Coverage Report](#coverage-report)
* [Integration Tests](#integration-tests)
* [Custom Test Patterns](#custom-test-patterns)

---

In [0]:
%pip install pytest pytest-cov pytest-mock neo4j networkx --quiet

import os
import subprocess

print("✅ Test dependencies installed successfully")
print("📦 Pytest version:", __import__('pytest').__version__)

In [0]:
import sys
from pathlib import Path

# ============================================================================
# PATH CONFIGURATION (Standard for all notebooks)
# ============================================================================

# Dynamic REPO_ROOT detection (works for any user)
try:
    # Try spark.conf first (works on Serverless)
    notebook_path = spark.conf.get("spark.databricks.notebook.path")
except:
    # Fallback to dbutils for classic compute
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except:
        # Last resort - use current working directory
        notebook_path = str(Path.cwd())
        print("⚠️  Could not detect notebook path, using current directory")

# Convert notebook path to workspace path and derive repo root
# notebook_path is like: /Users/{username}/skillup/evaluation/notebooks/NotebookName
if notebook_path.startswith("/"):
    workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
else:
    workspace_path = Path(notebook_path)

# Navigate up from notebooks -> evaluation -> skillup (repo root)
REPO_ROOT = workspace_path.parent.parent.parent if "notebooks" in str(workspace_path) else workspace_path

# Source data directory (version controlled)
DATA_DIR = REPO_ROOT / "data"

# Persistent artifact storage (Volumes - shared, not in git)
EVAL_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/evaluation")
DATA_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/data")

# Ensure artifact directories exist
EVAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Add skillup to Python path for imports
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"📁 REPO_ROOT: {REPO_ROOT}")
print(f"📁 DATA_DIR (source): {DATA_DIR}")
print(f"📦 EVAL_ARTIFACTS: {EVAL_ARTIFACTS}")
print(f"📦 DATA_ARTIFACTS: {DATA_ARTIFACTS}")

# Test directory
TESTS_DIR = REPO_ROOT / "tests"
print(f"🧪 TESTS_DIR: {TESTS_DIR}")

## Run All Unit Tests

Run the complete unit test suite with detailed verbose output.

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR), '-m', 'unit', '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

## Run Tests by Module

Execute tests for specific modules individually.

### Knowledge Graph Tests

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR / 'unit' / 'knowledgegraph'), '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

### Skill Gap Analysis Tests

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR / 'unit' / 'skillgap'), '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

### Recommender Tests

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR / 'unit' / 'recommender'), '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

### App Tests

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR / 'unit' / 'app'), '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

## Smoke Tests

Quick validation tests to verify critical functionality (< 30 seconds).

In [0]:
import time
start = time.time()
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR), '-m', 'smoke', '-v', '--tb=short'], env=env)
elapsed = time.time() - start
print(f"\n⏱️  Smoke tests completed in {elapsed:.2f} seconds")
if result.returncode != 0:
    print(f"⚠️ Tests exited with code {result.returncode}")

## HTML Coverage Report

Generate detailed HTML coverage report with line-by-line analysis.

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR), '-m', 'unit', '--cov=' + str(REPO_ROOT), '--cov-report=html', '--cov-report=term'], env=env)

print("\n" + "="*60)
if result.returncode == 0:
    print("✅ Coverage report generated!")
else:
    print("⚠️ Coverage report generated with test failures")
print("📊 View detailed report: htmlcov/index.html")
print("="*60)

## Integration Tests

Run integration tests that require external services (Neo4j, Databricks, OpenAI).

**Note:** These tests may be skipped if services are not configured.

In [0]:
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR), '-m', 'integration', '-v', '--runxfail'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

## Custom Test Patterns

Run tests matching a specific pattern or keyword.

Modify the pattern below to filter tests:

In [0]:
# Customize this pattern to match specific test names
test_pattern = "test_cv"  # Example: run all tests with "cv" in the name

print(f"🔍 Running tests matching pattern: '{test_pattern}'")
env = os.environ.copy()
env['PYTHONDONTWRITEBYTECODE'] = '1'
result = subprocess.run(['python', '-m', 'pytest', str(TESTS_DIR), '-k', test_pattern, '-v'], env=env)
if result.returncode != 0:
    print(f"\n⚠️ Tests exited with code {result.returncode}")

---

## Test Markers Reference

| Marker | Description |
|--------|-------------|
| `unit` | Fast, isolated unit tests |
| `integration` | Tests requiring external services |
| `smoke` | Critical functionality checks |
| `slow` | Tests taking significant time |
| `requires_neo4j` | Requires Neo4j connection |
| `requires_databricks` | Requires Databricks |
| `requires_openai` | Requires OpenAI API |

---

✅ Test execution complete!